In [11]:
import numpy as np
from scipy import stats

def compute_sample_size(p1, mde, alpha=0.05, power=0.80):
    """
    Compute required sample size per group for a two-proportion z-test.

    Parameters:
        p1    : baseline conversion rate (control)
        mde   : minimum detectable effect (absolute, e.g. 0.02 = 2pp)
        alpha : significance level (default 0.05)
        power : desired statistical power (default 0.80)

    Returns a dict with:
        - p2            : expected treatment rate (p1 + mde)
        - z_alpha       : critical value for significance
        - z_beta        : critical value for power
        - n_per_group   : required sample size per group (rounded up)
        - n_total       : total sample size across both groups
        - actual_mde_pct: the MDE expressed as % of baseline (relative)
    """
    assert 0 < p1 < 1
    assert mde > 0
    assert p1 + mde <= 1,  f"p2={p1+mde:.3f} exceeds 1.0"
    assert 0 < alpha < 1
    assert 0 < power < 1

    p2 = p1 + mde
    z_alpha = stats.norm.ppf(1- alpha/2)
    z_beta =  stats.norm.ppf(power)
    
    n_per_group = int(np.ceil((z_alpha + z_beta)**2 *(p1 *(1-p1) + p2* (1-p2))/mde**2))
    n_total= 2*n_per_group
    actual_mde_pct = mde/p1

    # ── Output validation ─────────────────────────────────
    assert 0 < p2 <= 1,                  "p2 must be in (0, 1]"
    assert z_alpha > 0,                  "z_alpha must be positive"
    assert z_beta > 0,                   "z_beta must be positive"
    assert n_per_group > 0,              "n_per_group must be positive"
    assert n_total == 2 * n_per_group,   "n_total must be 2x n_per_group"
    assert actual_mde_pct > 0,           "actual_mde_pct must be positive"

    return {'p2': p2,
        'z_alpha': z_alpha,
        'z_beta': z_beta,
        'n_per_group': n_per_group,
        'n_total': n_total,
        'actual_mde_pct': actual_mde_pct}



In [13]:
# ── Run it ──────────────────────────────────────────────
result = compute_sample_size(p1=0.22, mde=0.02)

for k, v in result.items():
    print(f"{k:>16}: {v}")

# ── Sensitivity check ────────────────────────────────────
print("\n── How does n change? ──")
for mde in [0.01, 0.02, 0.03, 0.05]:
    r = compute_sample_size(p1=0.22, mde=mde)
    print(f"MDE={mde:.2f} → n_per_group={r['n_per_group']:>7,}")

              p2: 0.24
         z_alpha: 1.959963984540054
          z_beta: 0.8416212335729143
     n_per_group: 6947
         n_total: 13894
  actual_mde_pct: 0.09090909090909091

── How does n change? ──
MDE=0.01 → n_per_group= 27,370
MDE=0.02 → n_per_group=  6,947
MDE=0.03 → n_per_group=  3,132
MDE=0.05 → n_per_group=  1,158


In [14]:
result = compute_sample_size(p1=0.22, mde=0.02, power =  0.90)
for k, v in result.items():
    print(f"{k:>16}: {v}")

              p2: 0.24
         z_alpha: 1.959963984540054
          z_beta: 1.2815515655446004
     n_per_group: 9300
         n_total: 18600
  actual_mde_pct: 0.09090909090909091
